In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

In [2]:
# Initialize Spark with Kafka and Postgres packages
spark = SparkSession.builder \
 .appName("FraudDetection") \
 .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
 .getOrCreate()
spark.sparkContext.setLogLevel("WARN")


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-72b68b7c-48f6-41d9-af21-4e9991d83d58;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1050ms :: artifacts dl 19ms
	:: modules in u

In [3]:
# 1. Load Static User Data (From CSV or Postgres)
# If using CSV: 
users_df = spark.read.csv("data/user_table.csv", header=True,inferSchema=True)

# 2. Read Streaming Data from Kafka
tx_schema = StructType([
 StructField("tx_id", IntegerType(), True),
 StructField("userId", IntegerType(), True),
 StructField("amount", DoubleType(), True),
 StructField("timestamp", DoubleType(), True)
])

kafka_stream = spark.readStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("subscribe", "fraud-alerts5") \
 .load()

26/06/19 06:17:03 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
# 3. Parse and Filter Data
parsed_stream = kafka_stream.select(from_json(col("value").cast("string"),
tx_schema).alias("tx")).select("tx.*")
# Fraud Logic: Amount > 5,000
fraud_stream = parsed_stream.filter(col("amount") > 5000.0)
fraud_stream2 = parsed_stream.filter(col("amount") > 10000.0)

# 4. Enrich Stream with User Details (Stream-Static Join)
enriched_fraud = fraud_stream.join(users_df, "userId")
enriched_fraud2 = fraud_stream2.join(users_df, "userId")

In [5]:
#5. Format for output Kafka topic
output_stream = enriched_fraud \
 .withColumn("value", to_json(struct("*")).cast("string")) \
 .select("value")
output_stream2 = enriched_fraud2 \
 .withColumn("value", to_json(struct("*")).cast("string")) \
 .select("value")

In [ ]:
# 6. Write Stream to 'fraud-notification' Topic
ry = output_stream.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-alertsss5k") \
 .option("checkpointLocation", "/workspace/checkpointssss") \
 .start()
ry2 = output_stream2.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-alertsss10k") \
 .option("checkpointLocation", "/workspace/checkpointsssss") \
 .start()
ry.awaitTermination()
ry2.awaitTermination()

26/06/19 06:19:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:19:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:19:32 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 5 : {fraud-alertsss5k=UNKNOWN_TOPIC_OR_PARTITION}
26/06/19 06:19:32 ERROR Metadata: [Producer clientId=producer-1] Metadata response reported invalid topics [fraud-alertsss>10k, fraud-alertsss>5k]
26/06/19 06:19:32 WARN NetworkClient: [Producer clientId=producer-1] Error while fetching metadata with correlation id 6 : {fraud-alertsss>10k=INVALID_TOPIC_EXCEPTION, fraud-alertsss10k=UNKNOWN_TOPIC_OR_PARTITION, fraud-alertsss>5k=INVALID_TOPIC_EXCEPTION}
26/06/19 06:19:32 ERROR Metadata: [Producer clientId=producer-1] Metadata response reported invalid topics [fraud-alertsss>10k, fraud-alert